In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path so `from tools...` imports work from any notebook subfolder.
_p = Path.cwd().resolve()
for _parent in [_p, *_p.parents]:
    if (_parent / 'tools' / 'search_tools.py').exists():
        sys.path.insert(0, str(_parent))
        break
del _p, _parent


In [16]:
%%capture --no-stderr
# %pip install "autogen-agentchat~=0.2.3"

# In Your OAI_CONFIG_LIST file, you must have two configs,
# one with:           "response_format": { "type": "text" }
# and the other with: "response_format": { "type": "json_object" }


[
    {"model": "gpt-4o-mini", "sk-REDACTED": "key go here", "response_format": {"type": "text"}},
]

In [1]:
import autogen
import os
from autogen.agentchat import UserProxyAgent
from autogen.agentchat.assistant_agent import AssistantAgent
from autogen.agentchat.groupchat import GroupChat
os.environ["SERPER_API_KEY"] = "1edefaec0732d11db50b993ba60539510cc55334"
from tools.search_tools import SearchTools




In [2]:
from autogen import ConversableAgent
from autogen import register_function

import os
import json
import requests

def search_internet(query: str) -> str:
        """Useful to search the internet
        about a a given topic and return relevant results"""
        print("Searching the internet...")
        top_result_to_return = 5
        url = "https://google.serper.dev/search"
        payload = json.dumps(
            {"q": query, "num": top_result_to_return, "tbm": "nws"})
        headers = {
            'X-API-KEY': os.environ['SERPER_API_KEY'],
            'content-type': 'application/json'
        }
        response = requests.request("POST", url, headers=headers, data=payload)
        # check if there is an organic key
        if 'organic' not in response.json():
            return "Sorry, I couldn't find anything about that, there could be an error with you serper api key."
        else:
            results = response.json()['organic']
            string = []
            print("Results:", results[:top_result_to_return])
            for result in results[:top_result_to_return]:
                try:
                    # Attempt to extract the date
                    date = result.get('date', 'Date not available')
                    string.append('\n'.join([
                        f"Title: {result['title']}",
                        f"Link: {result['link']}",
                        f"Date: {date}",  # Include the date in the output
                        f"Snippet: {result['snippet']}",
                        "\n-----------------"
                    ]))
                except KeyError:
                    next

            return '\n'.join(string)

        



In [ ]:
import asyncio
import autogen
import os
import httpx
from typing import Optional, List, Dict, Tuple, Union
import random  # noqa E402

import matplotlib.pyplot as plt  # noqa E402
import networkx as nx  # noqa E402

import autogen  # noqa E402
from autogen.agentchat.conversable_agent import ConversableAgent  # noqa E402
from autogen.agentchat.assistant_agent import AssistantAgent  # noqa E402
from autogen.agentchat.groupchat import GroupChat  # noqa E402
from autogen.graph_utils import visualize_speaker_transitions_dict 

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "sk-REDACTED"

# Define a custom HTTP client
class MyHttpClient(httpx.Client):
    def __deepcopy__(self, memo):
        return self
    
llm_config1 = {
    "config_list": [
        {
            "model": "nemotron",
            "api_type": "ollama",
            "client_host": "https://7r9o21n06von58-11434.proxy.runpod.net/",
        }
    ]
}




def is_termination_msg(content) -> bool:
    have_content = content.get("content", None) is not None
    if have_content and "TERMINATE" in content["content"]:
        return True
    return False


user_proxy = autogen.UserProxyAgent(
    name="User_proxy",
    system_message="A human admin who terminates the chat when the agent sends a message with 'TERMINATE' mentioned it it",
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config1,
)
AgentA = ConversableAgent(
    name="Character Developer",
    system_message=(
        "Role: Character Developer in a collaborative storytelling project.\n"
        "\n"
        "Context:\n"
        "You are responsible for creating engaging and multi-dimensional main characters for a short story OF 2000 WORDS.\n"
        "You are in a groupchat with the plot developer, world developer and your Leader. COLLABORATE WITH THEM TO WRITE THE STORY"
        "After finalising the initial things work towards finalising the story"
        "\n"
        # "Instructions:\n"
        # "- Develop detailed profiles for multiple main characters (names, backgrounds, personalities, motivations, flaws, growth arcs).\n"
        # "- Share character descriptions in the group chat.\n"
        # "- Ensure characters are suitable for the setting, plot, and themes proposed by others.\n"
        # "- **Be open to making significant changes** to your characters to achieve a cohesive story.\n"
        "- stick to the word limit strictly"
        "Include the keyword 'TERMINATE' at the end of your final version of the story** to signal the end of the group chat"
        "Note:\n"
        "- focus on compiling the story without overthinking and discussing about stuff.\n"
        "- do not schedule meetings and assume responses"
    ),
    llm_config=llm_config1,
)

AgentC = ConversableAgent(
    name="Plot Developer",
    system_message=(
        "Role: Plot Developer in a collaborative storytelling project.\n"
        "\n"
        "Context:\n"
        "You are tasked with developing the main plot for a short story of 2000 words\n"
        "You are in a groupchat with the character developer, plot developer, world developer and your Leader. COLLABORATE WITH THEM TO WRITE THE STORY"
        "After finalising the initial things work towards finalising the story"
        "\n"
        # "Instructions:\n"
        # "- Create an engaging main plot with a clear beginning, middle, and end.\n"
        # "- Introduce conflicts, challenges, and resolutions, including twists and turns.\n"
        # "- Share plot ideas in the group chat.\n"
        # "- **Be willing to revise significant plot points** for story cohesion.\n"
        "- stick to the word limit strictly"
        "Include the keyword 'TERMINATE' at the end of your final version of the story** to signal the end of the group chat"
        
        "\n"
        "Note:\n"
        "- focus on compiling the story without overthinking and discussing about stuff.\n"
        "- do not schedule meetings and assume responses"
    ),
    llm_config=llm_config1,
)



AgentB = ConversableAgent(
    name="Setting and World Developer",
    system_message=(
        "Role: Setting and World Developer in a collaborative storytelling project.\n"
        "\n"
        "Context:\n"
        "You are responsible for creating a rich and immersive setting for a multi-chapter short story of 2000 words\n"
        "You are in a groupchat with the character developer, plot developer and your Leader. COLLABORATE WITH THEM TO WRITE THE STORY"
        "After finalising the initial things work towards finalising the story"
        "\n"
        # "Instructions:\n"
        # "- Develop a vivid setting including time period, geography, culture, and societal structures.\n"
        # "- Create world-building elements such as history, politics, technology, and magic (if applicable).\n"
        # "- Share setting details in the group chat.\n"
        # "- **Be prepared to modify significant aspects** of the setting for coherence.\n"
        "- stick to the word limit strictly"
        "Include the keyword 'TERMINATE' at the end of your final version of the story** to signal the end of the group chat"
        
        # "\n"
        "Note:\n"
        "- focus on compiling the story without overthinking and discussing about stuff.\n"
        "- do not schedule meetings and assume responses"
    ),
    llm_config=llm_config1,
)

# AgentE = ConversableAgent(
#     name="Leader",
#     system_message=(
#         "Role: Head writer and leader overseeing the collaborative storytelling project.\n"
#         "\n"
#         "Context:\n"
#         "You are in a groupchat with the character developer, plot developer, world developer and theme and symbolism developer. Based on their inputs compile the versions of the story eventually finalising it."
#         "\n"
#         "Instructions:\n"
#         "- Compile versions of the story periodically.\n"
#         "- **Include the keyword 'TERMINATE' at the end of your final version of the story** to signal the end of the group chat.\n"
#         "- stick to the word limit strictly"
#         "\n"
#         "Note:\n"
#         "- focus on compiling the story without overthinking and discussing about stuff.\n"
#         "- do not schedule meetings and assume responses"
#     ),
#     llm_config=llm_config1,
# )


Agent5 = ConversableAgent(
    name="Tool_executor",
    system_message=(
        "you are responsible for running the tools"
    ),
    llm_config=llm_config1,
)




In [4]:
# import requests
# import json

# def query_ollama(prompt, model="qwen2.5:72b"):
#     url = "https://3s8bwscehzgw0q-11434.proxy.runpod.net/api/generate"  # Ensure correct endpoint
#     payload = {"model": model, "prompt": prompt}
    
#     try:
#         response = requests.post(url, json=payload)
#         response.raise_for_status()  # Check for HTTP errors
        
#         # Process response line by line
#         result = ""
#         for line in response.text.splitlines():
#             try:
#                 line_data = json.loads(line)
#                 result += line_data.get("response", "")
#                 if line_data.get("done", False):
#                     break
#             except json.JSONDecodeError:
#                 continue  # Ignore lines that aren't valid JSON
                
#         return result.strip()  # Return the concatenated response
#     except requests.exceptions.RequestException as e:
#         return {"error": "Request failed", "details": str(e)}

In [5]:
# # Add a global or class-level variable to track the first call
# is_first_call = True  # This flag tracks if the function is being called for the first time

# def custom_speaker_selection_func(last_speaker, groupchat):
#     global is_first_call

#     # If this is the first call, return the leader agent
#     if is_first_call:
#         is_first_call = False  # Reset the flag after the first call
#         print("First call detected. Setting speaker to Leader agent.")
#         for agent in groupchat.agents:
#             if agent.name == "Leader":  # Replace "Agent3" with the actual leader agent's name
#                 return agent
#         print("Error: Leader agent not found in the agents list.")
#         return None  # Handle the case where the leader agent is not found

#     # Access the last message in the group chat
#     last_message = groupchat.messages[-1]
#     print(f"Last message content: {last_message}")

#     # Prepare the input for the LLM
#     prompt = (
#     "You are a conversation coordinator. Based on the last message, decide which agent should speak next out of the following Summarizer_Agent_1, Summarizer_Agent_2, Summarizer_Agent_3 and Leader. "
#     "ONLY RESPOND WITH THE NAME OF THE AGENT AND NOTHING ELSE. NO OTHER CHARACTERS SHOULD BE THERE IN YOUR MESSAGE.\n\n"
#     f"The last message is: {last_message.get('content', '')}"
#     )

#     # Analyze the message using the local LLM
#     response = query_ollama(prompt)
#     print(f"LLM response: {response}")

#     # Extract the relevant text from the response dictionary
#     next_speaker_name = response  # Replace 'text' with the correct key

#     # Find the corresponding agent in the group chat
#     for agent in groupchat.agents:
#         if agent.name == next_speaker_name:
#             return agent

#     # If no valid agent is found, return None or a default fallback
#     print(f"No valid agent found for the name: {next_speaker_name}")
#     return None


In [6]:
# # Add global variables to track the first call and the call count
# is_first_call = True  # Tracks if this is the first call
# call_count = 0        # Tracks the number of times the function has been called

# def custom_speaker_selection_func(last_speaker, groupchat):
#     global is_first_call, call_count

#     # If this is the first call, return the leader agent
#     if is_first_call:
#         is_first_call = False  # Reset the flag after the first call
#         print("First call detected. Setting speaker to Leader agent.")

#         for agent in groupchat.agents:
#             if agent.name == "Leader":  # Replace "Leader" with the actual leader agent's name
#                 return agent
#         print("Error: Leader agent not found in the agents list.")
#         return None  # Handle the case where the leader agent is not found

#     # Increment the call count
#     call_count += 1

#     # If this is the 7th call, return the leader agent
#     if call_count % 7 == 0:
#         print(f"7th call detected (call count: {call_count}). Setting speaker to Leader agent.")
#         for agent in groupchat.agents:
#             if agent.name == "Leader":  # Replace "Leader" with the actual leader agent's name
#                 return agent
#         print("Error: Leader agent not found in the agents list.")
#         return None  # Handle the case where the leader agent is not found

#     # Access the last message in the group chat
#     last_message = groupchat.messages[-1]
#     print(f"Last message content: {last_message}")

#     # Prepare the input for the LLM
#     prompt = (
#         "You are a conversation coordinator. Based on the last message, decide which agent should speak next out of the following Production_Manager, Logistics_Manager, A/P—A/R_Manager, Customer_Relations/Marketing_Manager and Leader. "
#         "ONLY RESPOND WITH THE NAME OF THE AGENT AND NOTHING ELSE. NO OTHER CHARACTERS SHOULD BE THERE IN YOUR MESSAGE.\n\n"
#         f"The last message is: {last_message.get('content', '')}"
#     )

#     # Analyze the message using the local LLM
#     response = query_ollama(prompt)
#     print(f"LLM response: {response}")

#     # Extract the relevant text from the response dictionary
#     next_speaker_name = response.strip()  # Use .strip() to remove any extra spaces or newlines

#     # Find the corresponding agent in the group chat
#     for agent in groupchat.agents:
#         if agent.name == next_speaker_name:
#             return agent

#     # If no valid agent is found, return None or a default fallback
#     print(f"No valid agent found for the name: {next_speaker_name}")
#     return None


In [ ]:
from autogen.graph_utils import visualize_speaker_transitions_dict
# Define your agents
agents = [AgentA, AgentB, AgentC, user_proxy, Agent5]

# # Initialize the allowed speaker transitions dictionary
# allowed_speaker_transitions_dict = {}

# # Set up transitions for each agent
# for agent in agents:
#     if agent == Agent5:
#         # Agent5 cannot send messages to any agent
#         allowed_speaker_transitions_dict[agent] = []
#     else:
#         # Other agents can send messages to all agents except themselves and Agent5
#         allowed_speaker_transitions_dict[agent] = [
#             other_agent for other_agent in agents
#             if other_agent != agent and other_agent != Agent5
#         ]

# # Allow user_proxy to send messages to Agent5
# allowed_speaker_transitions_dict[user_proxy].append(Agent5)

# # Visualize the transitions
# visualize_speaker_transitions_dict(allowed_speaker_transitions_dict, agents)


In [ ]:
def is_termination_msg(content) -> bool:
    have_content = content.get("content", None) is not None
    if have_content and "TERMINATE" in content["content"]:
        return True
    return False

group_chat = GroupChat(
    agents=agents,
    messages=[],
    max_round=100,
    # allowed_or_disallowed_speaker_transitions=allowed_speaker_transitions_dict,
    # speaker_transitions_type="allowed",
)
# Create the manager
manager = autogen.GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config1,
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    code_execution_config=False,
)


In [9]:
# from autogen import register_function


# register_function(
#     search_internet,
#     caller=Leader,  # The assistant agent can suggest calls to the calculator.
#     executor=Agent5,  # The user proxy agent can execute the calculator calls.
#     name="search",  # By default, the function name is used as the tool name.
#     description="browse the internet for information using this tool",  # A description of the tool
# )

# register_function(
#     search_internet,
#     caller=Agent1,  # The assistant agent can suggest calls to the calculator.
#     executor=Agent5,  # The user proxy agent can execute the calculator calls.
#     name="search",  # By default, the function name is used as the tool name.
#     description="browse the internet for information using this tool",  # A description of the tool
# )

# register_function(
#     search_internet,
#     caller=Agent2,  # The assistant agent can suggest calls to the calculator.
#     executor=Agent5,  # The user proxy agent can execute the calculator calls.
#     name="search",  # By default, the function name is used as the tool name.
#     description="browse the internet for information using this tool",  # A description of the tool
# )

# register_function(
#     search_internet,
#     caller=Agent3,  # The assistant agent can suggest calls to the calculator.
#     executor=Agent5,  # The user proxy agent can execute the calculator calls.
#     name="search",  # By default, the function name is used as the tool name.
#     description="browse the internet for information using this tool",  # A description of the tool
# )

# register_function(
#     search_internet,
#     caller=Agent4,  # The assistant agent can suggest calls to the calculator.
#     executor=Agent5,  # The user proxy agent can execute the calculator calls.
#     name="search",  # By default, the function name is used as the tool name.
#     description="browse the internet for information using this tool",  # A description of the tool
# )

In [10]:
# chat_result = user_proxy.initiate_chat(Agent0, message="search internet about google. Use production Manager first")

In [11]:
# Prepare the initial message with the novel's text
initial_message = (
    "please Create an adventure thriller short story of about 2000 words"
)

# Initiate the conversation
user_proxy.initiate_chat(manager, message=initial_message)

User_proxy (to chat_manager):

please Create an adventure thriller short story of about 2000 words

--------------------------------------------------------------------------------

Next speaker: Plot Developer


>>>>>>>> USING AUTO REPLY...
Plot Developer (to chat_manager):

Given the collaborative nature of this project, I'll outline my contributions as the **Plot Developer**. Since I don't have real-time feedback from the Character Developer, World Developer, and Leader in this format, I'll make assumptions based on common adventure thriller elements. Please find below the plot development, along with suggested inputs from other team members (in brackets):

---

**Story Title:** The Echoes of Arkon

**Genre:** Adventure Thriller

**Word Count Goal:** 2000 words

### **Plot Outline:**

#### Act I: Setup (~300 words)

1. **Introduction to Protagonist** [**Character Developer Input Needed:**
   - **Name:** Alex Chen
   - **Background:** Former Military, turned Archaeologist
   - **Moti

ChatResult(chat_id=None, chat_history=[{'content': 'please Create an adventure thriller short story of about 2000 words', 'role': 'assistant', 'name': 'User_proxy'}, {'content': "Given the collaborative nature of this project, I'll outline my contributions as the **Plot Developer**. Since I don't have real-time feedback from the Character Developer, World Developer, and Leader in this format, I'll make assumptions based on common adventure thriller elements. Please find below the plot development, along with suggested inputs from other team members (in brackets):\n\n---\n\n**Story Title:** The Echoes of Arkon\n\n**Genre:** Adventure Thriller\n\n**Word Count Goal:** 2000 words\n\n### **Plot Outline:**\n\n#### Act I: Setup (~300 words)\n\n1. **Introduction to Protagonist** [**Character Developer Input Needed:**\n   - **Name:** Alex Chen\n   - **Background:** Former Military, turned Archaeologist\n   - **Motivation:** Uncover the truth behind her father's disappearance]\n   \n2. **Incitin

In [12]:
last_message = group_chat.messages[-1] if group_chat.messages else None
if last_message:
    print("Final Message Content:", last_message['content'])

Final Message Content: **COMPILED STORY (VERSION 1.3 - FINAL)**


**ELYRIA'S SPIRE**

**World-Building:**

* **Stellar Scribes' Legacy:** Ancient order of scholars, inventors, and explorers who unraveled Eldridian secrets.
	+ **Echoing Tomes:** Cryptic journals containing Eldridian knowledge and warnings about the spire's power.
	+ **Stellar Key:** Lost device capable of unlocking hidden pathways within Elyria's Spire.
* **Enhanced Spire Architecture:**
	1. **Chamber of Harmony:** Represents balance, where Eldridian technology synchronizes with the planet's energy.
	2. **Chamber of Reflection:** Embodies responsibility, containing ancient wisdom and warnings about the spire's power.
	3. **Chamber of Ascension:** Symbolizes protagonists' growth, where they apply learned lessons to unlock the spire's true potential.
* **Symbolic Elements:**
	+ **Luminous Core:** Glowing, crystalline structure reflecting the balance between Elyria's elements.
	+ **Ouroboros Mural:** Circular artwork depic

In [13]:
def save_conversation_to_file(groupchat, filename="chat.txt"):
    """
    Save the entire conversation history to a specified file.

    Args:
        groupchat (GroupChat): The GroupChat instance containing the messages.
        filename (str): The name of the file to save the conversation history.
    """
    if not groupchat.messages:
        print("No messages in the group chat to save.")
        return

    # Compile the conversation history
    conversation_history = "\n".join(
        f"{msg['role']}: {msg['content']}" for msg in groupchat.messages
    )

    # Write the conversation history to the file
    with open(filename, "w", encoding="utf-8") as file:
        file.write(conversation_history)

    print(f"Conversation history saved to {filename}")
    
save_conversation_to_file(group_chat, filename="chat.txt")


Conversation history saved to chat.txt


In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# Initialize the Llama 3.1 model
llm = ChatOpenAI(
    model="llama3.1",
    base_url="http://44.221.48.158:11434/v1"
)

def structure_logs_with_local_llm(file_path, initial_message):
    """
    Reads chat logs from a file and generates a structured summary.

    Args:
        file_path (str): Path to the chat log file.
        initial_message (str): The initial task or prompt for context.

    Returns:
        str: The structured summary generated by the LLM.
    """
    # Read the chat logs from the file
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            log_context = file.read()
    except FileNotFoundError:
        return "Error: The specified file was not found."
    except Exception as e:
        return f"An error occurred while reading the file: {e}"

    # Prepare the messages for the LLM
    messages = [
        {
            "role": "system",
            "content": (
                "You are a professional chat summarizer who goes through the entire chat and creates a proper summary based on the '{initial_message}'."
            )
        },
        {
            "role": "user",
            "content": (
                f"Convert the following agent logs into a structured format and into a proper summarized final output "
                f"based on the task '{initial_message}'.\n\nLogs:\n{log_context}"
            )
        }
    ]

    # Generate the structured summary using the LLM
    try:
        response = llm.invoke(messages)
        if isinstance(response, AIMessage):
            structured_summary = response.content
        else:
            structured_summary = "Unexpected response type from the model."
    except Exception as e:
        structured_summary = f"An error occurred during LLM processing: {e}"

    return structured_summary


In [15]:
# Define the path to your chat log file and the initial task message
file_path = 'chat.txt'
initial_message = 'Design a comprehensive digital marketing course.'

# Generate the structured summary
summary = structure_logs_with_local_llm(file_path, initial_message)

# Output the summary
print(summary)

An error occurred during LLM processing: Request timed out.
